# Rolling pair selection and training-period backtest — zero transaction costs

This notebook develops the strategy **only inside the training partition**. The final three calendar years are an untouched validation holdout (also called the future test set).

Every two months, use the preceding **12 calendar months** to screen same-industry-group pairs, estimate their log-price relationship, and freeze the model for the next **two calendar months** of trading. These trading windows are out of sample relative to their own estimates, but remain part of strategy development—not final holdout evidence.

Workflow: training-only prices → rolling schedule → return correlations → bidirectional Engle–Granger tests → frozen spread parameters → independent pair accounts → continuous portfolio performance.

Transaction costs: **0 bps per leg at entry and exit**. This zero-cost scenario recalculates holdings, equity, and performance with costs disabled. Outputs are separate from the 5 bps scenario; the historical Treasury benchmark still applies to Sharpe.


## 1. Configuration and imports

Dates are explicit dataset boundaries from notebook 1, rather than inferred by examining holdout prices. All intervals use **inclusive start, exclusive end**. With the current snapshot, training ends before August 25, 2023 and validation is reserved through August 24, 2026. Changing the dataset snapshot requires deliberately updating these boundaries.

All strategy settings live in this cell. No parameter tuning or validation execution occurs here.

In [ ]:
import csv
import sys
import json
import warnings
from inspect import signature
from itertools import combinations
from pathlib import Path
from importlib.metadata import version

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pandas_market_calendars as mcal
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller, coint
from IPython.display import display

PROJECT_FOLDER = Path.cwd()
if PROJECT_FOLDER.name == "notebooks":
    PROJECT_FOLDER = PROJECT_FOLDER.parent
sys.path.insert(0, str(PROJECT_FOLDER / "src"))
from risk_free import annualized_treasury_return, load_treasury_yields

DATA_FOLDER = PROJECT_FOLDER / "data" / "Filter_historical"
RAW_DATA_FOLDER = PROJECT_FOLDER / "data" / "historical"
GICS_FILE = PROJECT_FOLDER / "data" / "gics" / "ticker_gics.csv"
OUTPUT_ROOT = PROJECT_FOLDER / "outputs" / "rolling_training_zero_cost"

DATA_START = pd.Timestamp("2015-01-01")
DATA_END_EXCLUSIVE = pd.Timestamp("2026-08-25")
HOLDOUT_START = DATA_END_EXCLUSIVE - pd.DateOffset(years=3)
FORMATION_MONTHS = 12
TRADING_MONTHS = 2
CORRELATION_THRESHOLD = 0.8
EG_SIGNIFICANCE = 0.05
ENTRY_Z, EXIT_Z, STOP_Z = 1.5, 0.5, 3.0
TRANSACTION_COST_BPS = 0.0
INITIAL_CAPITAL = 100_000.0
TRADING_DAYS = 252

assert np.isfinite([ENTRY_Z, EXIT_Z, STOP_Z]).all()
assert 0 <= EXIT_Z < ENTRY_Z < STOP_Z
assert np.isfinite(TRANSACTION_COST_BPS) and TRANSACTION_COST_BPS >= 0
assert np.isfinite(INITIAL_CAPITAL) and INITIAL_CAPITAL > 0
assert EG_SIGNIFICANCE == 0.05  # The EG test below uses the returned 5% critical value.
assert DATA_START + pd.DateOffset(months=FORMATION_MONTHS) < HOLDOUT_START

## 2. Load only training observations

The source CSVs mix training and holdout dates. The reader checks each date and discards rows outside training **before converting or storing their price field**. It necessarily scans CSV text, but never constructs, analyzes, displays, or uses holdout prices. Do not replace this with a full-price load followed by a late split.

Use the NYSE session calendar to expose missing training observations. Do not forward-fill, backward-fill, or interpolate prices. Eligibility is reassessed from each formation window alone; neither future availability nor full-history coverage determines selection.

In [ ]:
def read_training_series(path, start=DATA_START, end=HOLDOUT_START):
    """Date-gate mixed CSV rows before interpreting adjusted prices."""
    if not DATA_START <= start < end <= HOLDOUT_START:
        raise ValueError("Price loading must stay inside the training partition")
    observations = {}
    with path.open(newline="") as stream:
        reader = csv.DictReader(stream)
        if not {"Date", "Adj Close"}.issubset(reader.fieldnames or []):
            raise ValueError(f"Missing Date / Adj Close columns: {path.name}")
        for row in reader:
            date = pd.Timestamp(row["Date"])
            if pd.isna(date):
                raise ValueError(f"Missing date: {path.name}")
            if not start <= date < end:
                continue
            if date in observations:
                raise ValueError(f"Duplicate training date: {path.name}, {date}")
            value = pd.to_numeric(row["Adj Close"], errors="coerce")
            observations[date] = float(value)
    return pd.Series(observations, dtype=float, name=path.stem).sort_index()


calendar = mcal.get_calendar("NYSE")
sessions = calendar.valid_days(DATA_START, HOLDOUT_START - pd.Timedelta(days=1))
sessions = sessions.tz_localize(None)
files = sorted(DATA_FOLDER.glob("*.csv"))
if not files:
    raise FileNotFoundError(DATA_FOLDER)
training_prices = pd.concat([read_training_series(path) for path in files], axis=1)
training_prices = training_prices.reindex(sessions).rename_axis("Date")
assert training_prices.index.max() < HOLDOUT_START
print(f"Training only: {training_prices.index.min().date()} to "
      f"{training_prices.index.max().date()}, {len(training_prices):,} sessions")
print(f"Price files: {len(files)}; validation prices excluded at ingestion")

### Industry groups

Use the supplied **four-digit Industry Group Code**, not the eight-digit sub-industry code. Codes are strings so their hierarchy remains explicit. The supplied mapping is static; it does not establish point-in-time industry membership.

In [ ]:
gics = pd.read_csv(GICS_FILE, dtype="string")
gics["Ticker"] = gics["Ticker"].str.strip().str.upper()
gics["Industry Group Code"] = gics["Industry Group Code"].str.strip()
if gics["Ticker"].duplicated().any():
    raise ValueError("Duplicate tickers in GICS mapping")
valid_code = gics["Industry Group Code"].str.fullmatch(r"\d{4}", na=False)
gics = gics.loc[valid_code & gics["Ticker"].isin(training_prices.columns)].copy()
unmapped = sorted(set(training_prices.columns) - set(gics["Ticker"]))
print(f"Mapped training columns: {len(gics)}; excluded without a valid group: {len(unmapped)}")
display(gics.groupby(["Industry Group Code", "Industry Group"]).size().rename("Stocks").to_frame())

## 3. Define the rolling windows

The first formation interval is January–December 2015, followed by January–February 2016 trading. The next uses March 2015–February 2016 to trade March–April 2016. Formation windows overlap; trading windows do not. Observations from an earlier trading window can enter later estimation once historical.

Include the final shortened trading interval, explicitly flagged, so the curve reaches the training boundary. The initial 12-month estimation interval has no strategy return and is not included in annualization.

In [ ]:
def make_windows(start, end):
    rows = []
    trade_start = start + pd.DateOffset(months=FORMATION_MONTHS)
    while trade_start < end:
        scheduled_end = trade_start + pd.DateOffset(months=TRADING_MONTHS)
        trade_end = min(scheduled_end, end)
        rows.append({
            "Window": len(rows) + 1,
            "Formation Start": trade_start - pd.DateOffset(months=FORMATION_MONTHS),
            "Formation End Exclusive": trade_start,
            "Trading Start": trade_start,
            "Trading End Exclusive": trade_end,
            "Shortened": trade_end < scheduled_end,
        })
        trade_start = scheduled_end
    return pd.DataFrame(rows)


def interval(data, start, end):
    return data.loc[(data.index >= start) & (data.index < end)]


windows = make_windows(DATA_START, HOLDOUT_START)
assert (windows["Trading End Exclusive"] <= HOLDOUT_START).all()
assert (windows["Formation End Exclusive"] == windows["Trading Start"]).all()
assert (windows["Trading Start"].iloc[1:].to_numpy()
        == windows["Trading End Exclusive"].iloc[:-1].to_numpy()).all()
display(windows)

## 4. Screen daily log-return correlations

Within each formation window, a stock must have finite, positive adjusted prices on **every NYSE session** and nonconstant prices. This complete-history rule is evaluated without looking at trading-window data. Rejections remain in an eligibility audit.

Compute `log(price).diff()` on the session grid before comparing any pair; the first formation date has no within-window return. Compare every unique pair in each four-digit group, recording undefined correlations and ineligible pairs too. Only Pearson correlations **strictly greater than 0.8** proceed to cointegration testing.

In [ ]:
def screen_correlations(formation, mapping):
    valid = np.isfinite(formation) & formation.gt(0)
    eligibility = pd.DataFrame({
        "Valid Sessions": valid.sum(),
        "Required Sessions": len(formation),
        "Complete Positive History": valid.all(),
        "Nonconstant": formation.nunique() > 1,
    })
    eligibility["Eligible"] = (eligibility["Complete Positive History"]
                                & eligibility["Nonconstant"])
    eligibility.index.name = "Ticker"
    eligible = eligibility.index[eligibility["Eligible"]]
    log_returns = np.log(formation.loc[:, eligible]).diff().iloc[1:]
    rows = []
    for group_code, group in mapping.groupby("Industry Group Code", sort=True):
        tickers = sorted(group["Ticker"])
        eligible_tickers = [ticker for ticker in tickers if ticker in eligible]
        correlations = log_returns[eligible_tickers].corr()
        for a, b in combinations(tickers, 2):
            ready = a in eligible and b in eligible
            corr = correlations.loc[a, b] if ready else np.nan
            rows.append({
                "Industry Group Code": group_code,
                "Industry Group": group["Industry Group"].iloc[0],
                "Ticker 1": a, "Ticker 2": b,
                "Return Observations": len(log_returns) if ready else 0,
                "Correlation": corr,
                "Screen Passed": bool(np.isfinite(corr) and corr > CORRELATION_THRESHOLD),
                "Status": ("OK" if np.isfinite(corr) else "Undefined correlation")
                          if ready else "Incomplete or constant formation history",
            })
    columns = ["Industry Group Code", "Industry Group", "Ticker 1", "Ticker 2",
               "Return Observations", "Correlation", "Screen Passed", "Status"]
    return pd.DataFrame(rows, columns=columns), eligibility.reset_index()

## 5. Bidirectional Engle–Granger selection

For every correlation-screened pair, fit both directions with an intercept:

\[
\log P_A=\alpha+\beta\log P_B+\epsilon.
\]

Use augmented Engle–Granger with AIC lag selection and the default maximum lag. The residual ADF regression has **no added constant** after fitting the intercept in the first stage. Selection compares the statistic against the returned **5% EG-specific critical value**, not ordinary ADF critical values or p-values. The EG p-value is retained for ranking directions. See the [statsmodels EG reference](https://www.statsmodels.org/stable/generated/statsmodels.tsa.stattools.coint.html).

A pair qualifies if either direction passes. If both pass, keep the lower EG p-value (ticker order breaks ties). Each unordered pair receives only one capital allocation. Store both directions, including rejected or numerically degenerate fits. There is no fixed number of selected pairs and no additional performance ranking.

### Frozen spread parameters

For the selected direction, define \(s=\log P_A-\beta\log P_B\). Estimate its mean and **sample** standard deviation from the same 12 months. Trading uses \(z=(s-\mu)/\sigma\). Freeze beta, mean, and standard deviation for the entire trading window. Alpha is a regression diagnostic; subtracting the training mean already centers the spread.

In [ ]:
FIT_COLUMNS = ["Industry Group Code", "Industry Group", "Ticker 1", "Ticker 2",
               "Correlation", "A", "B", "Observations", "Alpha", "Beta",
               "Spread Mean", "Spread Std", "EG Statistic", "EG p-value",
               "EG 5% Critical Value", "ADF Lags", "ADF Observations",
               "EG Passed", "Chosen", "Status"]


def fit_direction(log_prices, a, b):
    """Estimate OLS and a calibrated EG test on the identical formation sample."""
    fit = sm.OLS(log_prices[a], sm.add_constant(log_prices[[b]], has_constant="add")).fit()
    alpha, beta = float(fit.params["const"]), float(fit.params[b])
    spread = log_prices[a] - beta * log_prices[b]
    mean, std = float(spread.mean()), float(spread.std(ddof=1))
    if not np.isfinite([alpha, beta, mean, std]).all() or std <= 1e-12:
        raise ValueError("Degenerate spread or nonfinite model")
    with warnings.catch_warnings():
        warnings.simplefilter("error")
        statistic, pvalue, critical = coint(
            log_prices[a], log_prices[b], trend="c", method="aeg",
            maxlag=None, autolag="aic", return_results=False,
        )
    # Retain lag diagnostics only; ordinary ADF p-values/critical values are not used.
    adf_options = {"result_object": False} if "result_object" in signature(adfuller).parameters else {}
    adf = adfuller(fit.resid, regression="n", maxlag=None, autolag="AIC", **adf_options)
    if not np.isfinite([statistic, pvalue, critical[1]]).all():
        raise ValueError("Nonfinite EG result")
    np.testing.assert_allclose(statistic, adf[0], rtol=1e-8, atol=1e-8)
    return {
        "Alpha": alpha, "Beta": beta, "Spread Mean": mean, "Spread Std": std,
        "EG Statistic": statistic, "EG p-value": pvalue,
        "EG 5% Critical Value": critical[1], "ADF Lags": adf[2],
        "ADF Observations": adf[3], "EG Passed": bool(statistic < critical[1]),
        "Status": "OK",
    }


def select_pairs(formation, comparisons):
    rows = []
    for _, pair in comparisons.loc[comparisons["Screen Passed"]].iterrows():
        a, b = pair["Ticker 1"], pair["Ticker 2"]
        logs = np.log(formation[[a, b]])
        for dependent, independent in [(a, b), (b, a)]:
            row = {key: pair[key] for key in FIT_COLUMNS[:5]}
            row.update({"A": dependent, "B": independent, "Observations": len(logs),
                        "EG Passed": False, "Chosen": False})
            try:
                row.update(fit_direction(logs, dependent, independent))
            except (ValueError, np.linalg.LinAlgError, Warning) as error:
                row["Status"] = str(error)
            rows.append(row)
    directions = pd.DataFrame(rows, columns=FIT_COLUMNS)
    passed = directions.loc[directions["EG Passed"].eq(True)]
    chosen = (passed.sort_values(["EG p-value", "A", "B"])
              .drop_duplicates(["Ticker 1", "Ticker 2"]).index)
    directions.loc[chosen, "Chosen"] = True
    return directions, directions.loc[chosen].sort_values(["A", "B"]).copy()

## 6. Trading rules and accounting

Signals observed at a close execute at the **next session's close**. The final formation close may generate an entry for the first trading close. Entries earn no return before execution. Execution uses the pending signal, even if the next close has moved; the new close then generates the next decision. A stop is consequently a trigger, not a guaranteed execution price or loss limit.

| State | Signal at close | Order for next close |
| --- | --- | --- |
| Flat, eligible | `−STOP_Z < z ≤ −ENTRY_Z` | Long spread |
| Flat, eligible | `ENTRY_Z ≤ z < STOP_Z` | Short spread |
| Long spread | `z ≥ −EXIT_Z` | Mean-reversion exit |
| Short spread | `z ≤ EXIT_Z` | Mean-reversion exit |
| Long spread | `z ≤ −STOP_Z` | Stop and blacklist |
| Short spread | `z ≥ STOP_Z` | Stop and blacklist |

A stop permanently disables entries for the remaining window. Ordinary exits permit later re-entry. A flat pair already beyond the stop band cannot enter, but is not blacklisted merely for being flat outside that band. Every window starts with a fresh blacklist. At its final close, scheduled liquidation overrides pending orders and prevents new entries.

### Capital and costs

Allocate current portfolio capital equally among selected pairs at each window boundary. Keep independent accounts; idle or blacklisted capital stays in that account. At entry, signed dollar weights are `direction × [1, −beta] / (1 + abs(beta))`. Gross entry notional is account equity divided by `1 + cost_rate`, so entry costs plus gross notional fit the allocation. Hold adjusted-price units until exit; fixed beta does not imply daily dollar-weight rebalancing.

Charge **`TRANSACTION_COST_BPS` basis points per leg's absolute traded value** on entry and exit. Cash plus marked holdings determines equity, including short-sale proceeds and liabilities. No selected pairs means cash with zero return. Unexpected missing trading prices cause a visible failure: no future-availability filtering or silent shortening is allowed.

In [ ]:
def decide_order(direction, z, blacklisted):
    """Return the next-close target and the persistent blacklist state."""
    if direction:
        if direction * z <= -STOP_Z:
            return (0, "Stop loss"), True
        if direction * z >= -EXIT_Z:
            return (0, "Mean reversion"), blacklisted
    elif not blacklisted and ENTRY_Z <= abs(z) < STOP_Z:
        return (-1 if z > 0 else 1, "Entry"), blacklisted
    return None, blacklisted


TRADE_COLUMNS = ["A", "B", "Entry Signal Date", "Entry Date", "Direction",
                 "Entry Signal z", "Entry z", "Entry Equity ($)", "Entry Cost ($)",
                 "Exit Signal Date", "Exit Date", "Exit Signal z", "Exit z",
                 "Exit Reason", "Net P/L ($)", "Net Return", "Costs ($)"]


def backtest_pair(pair, trading, formation_last, initial_capital,
                  cost_bps=None):
    """Run one frozen model with lagged signals, fixed holdings, and a blacklist."""
    if cost_bps is None:
        cost_bps = TRANSACTION_COST_BPS
    a, b = pair["A"], pair["B"]
    beta, mean, std = (float(pair[key]) for key in ["Beta", "Spread Mean", "Spread Std"])
    if initial_capital <= 0 or cost_bps < 0 or std <= 0:
        raise ValueError("Invalid capital, cost, or spread scale")
    data = trading[[a, b]]
    if data.empty or not (np.isfinite(data) & data.gt(0)).all().all():
        raise ValueError(f"{a}/{b}: incomplete or invalid trading prices")
    z = (np.log(data[a]) - beta * np.log(data[b]) - mean) / std
    previous_z = float((np.log(formation_last[a]) - beta * np.log(formation_last[b]) - mean) / std)
    previous_date = formation_last.name
    pending, blacklisted = decide_order(0, previous_z, False)
    cash = equity = initial_capital
    units = np.zeros(2)
    direction, active_trade = 0, None
    rows, trades = [], []
    rate = cost_bps / 10_000

    for i, (date, prices) in enumerate(data.iterrows()):
        px = prices.to_numpy(dtype=float)
        prior_equity = equity
        before_trade = cash + units @ px
        if before_trade <= 0:
            raise ValueError(f"{a}/{b}: capital exhausted on {date}")
        cost, traded_value, reason = 0.0, 0.0, ""
        action = pending
        final_close = i == len(data) - 1
        if final_close:
            action = (0, "End of window") if direction else None
        if action is not None:
            target, reason = action
            if target == 0 and direction:
                traded_value = float(np.abs(units * px).sum())
                cost = rate * traded_value
                cash += float(units @ px) - cost
                units = np.zeros(2)
                active_trade.update({
                    "Exit Signal Date": pd.NaT if final_close else previous_date,
                    "Exit Date": date,
                    "Exit Signal z": np.nan if final_close else previous_z,
                    "Exit z": z.loc[date], "Exit Reason": reason,
                    "Net P/L ($)": cash - active_trade["Entry Equity ($)"],
                    "Costs ($)": active_trade["Entry Cost ($)"] + cost,
                })
                active_trade["Net Return"] = (active_trade["Net P/L ($)"]
                                               / active_trade["Entry Equity ($)"])
                trades.append(active_trade)
                active_trade, direction = None, 0
            elif target and direction == 0:
                weights = target * np.array([1.0, -beta]) / (1 + abs(beta))
                units = before_trade / (1 + rate) * weights / px
                traded_value = float(np.abs(units * px).sum())
                cost = rate * traded_value
                cash -= float(units @ px) + cost
                direction = target
                active_trade = {
                    "A": a, "B": b, "Entry Signal Date": previous_date,
                    "Entry Date": date, "Direction": direction,
                    "Entry Signal z": previous_z, "Entry z": z.loc[date],
                    "Entry Equity ($)": before_trade, "Entry Cost ($)": cost,
                }
        equity = float(cash + units @ px)
        if equity <= 0:
            raise ValueError(f"{a}/{b}: capital exhausted after costs on {date}")
        pending = None
        if not final_close:
            pending, blacklisted = decide_order(direction, float(z.loc[date]), blacklisted)
        rows.append({
            "Date": date, "z": z.loc[date], "Position": direction,
            "Blacklisted": blacklisted, "Equity": equity,
            "Net Return": equity / prior_equity - 1,
            "Before Trade Equity": before_trade,
            "Costs ($)": cost, "Traded Value ($)": traded_value,
            "Turnover": traded_value / before_trade, "Action": reason,
            "Units A": units[0], "Units B": units[1],
        })
        previous_date, previous_z = date, float(z.loc[date])
    daily = pd.DataFrame(rows).set_index("Date")
    daily["Gross Equity"] = daily["Equity"] + daily["Costs ($)"].cumsum()
    return daily, pd.DataFrame(trades, columns=TRADE_COLUMNS)

## 7. Aggregate independent accounts and calculate metrics

Sum account equities, actual costs, and traded notional. Allocations are equal **at window inception**, then drift with account performance; do not average pair returns each day, which would imply daily capital transfers. Portfolio turnover is daily total absolute traded notional divided by portfolio equity immediately before that day's transactions.

All main metrics are net of costs. Annualization uses 252 trading sessions; volatility uses the daily sample standard deviation; Sharpe is `(Annualized Return - Annualized Risk-Free Return) / Annualized Volatility`. The benchmark uses historical [FRED DGS3MO](https://fred.stlouisfed.org/series/DGS3MO) three-month Treasury investment-basis yields, cached in `data/risk_free/DGS3MO.csv`. Each trading session uses the latest yield strictly before that date (carried forward at most seven calendar days), converted to an effective daily return as `(1 + yield / 100) ** (1 / 252) - 1`. These returns are compounded over the exact window or full backtest and annualized with 252 sessions/year. This approximates rolling bills from yields; it is not a realized Treasury total-return index. Full-period benchmark returns are recomputed over the full period, never averaged across windows. Missing coverage stops execution; delete the CSV and restart the kernel to refresh it. Drawdown includes the initial capital peak. Trade statistics refer to completed round trips, including final liquidation, and include both entry and exit costs. Undefined ratios (for example Sharpe with zero volatility) are `NaN`.

“Gross equity” adds actual cash costs back to the same executed path. It is a cost-attribution diagnostic, not a separately resized zero-cost strategy. Two-month annualized metrics can be unstable; inspect realized returns and trade counts alongside them.

In [ ]:
def combine_accounts(accounts, dates, initial_capital):
    columns = ["Equity", "Before Trade Equity", "Costs ($)", "Traded Value ($)"]
    if accounts:
        daily = sum((account[columns] for account in accounts))
        daily["Active Pairs"] = sum((account["Position"].ne(0).astype(int) for account in accounts))
        daily["Blacklisted Pairs"] = sum((account["Blacklisted"].astype(int) for account in accounts))
    else:
        daily = pd.DataFrame(0.0, index=dates, columns=columns)
        daily[["Equity", "Before Trade Equity"]] = initial_capital
        daily[["Active Pairs", "Blacklisted Pairs"]] = 0
    previous = daily["Equity"].shift(1, fill_value=initial_capital)
    daily["Net Return"] = daily["Equity"] / previous - 1
    daily["Turnover"] = daily["Traded Value ($)"] / daily["Before Trade Equity"]
    daily["Gross Equity"] = daily["Equity"] + daily["Costs ($)"].cumsum()
    daily.index.name = "Date"
    return daily


def performance_metrics(daily, trades, initial_capital, treasury_yields=None):
    returns = daily["Net Return"]
    volatility = returns.std(ddof=1)
    cumulative = daily["Equity"].iloc[-1] / initial_capital - 1
    peak = daily["Equity"].cummax().clip(lower=initial_capital)
    annualized_return = (1 + cumulative) ** (TRADING_DAYS / len(daily)) - 1
    annualized_volatility = volatility * np.sqrt(TRADING_DAYS)
    if treasury_yields is None:
        treasury_yields = load_treasury_yields(PROJECT_FOLDER / "data" / "risk_free" / "DGS3MO.csv")
    risk_free_rate = annualized_treasury_return(daily.index, treasury_yields, TRADING_DAYS)
    return {
        "Cumulative Return": cumulative,
        "Gross Cumulative Return": daily["Gross Equity"].iloc[-1] / initial_capital - 1,
        "Annualized Return": annualized_return,
        "Annualized Volatility": annualized_volatility,
        "Annualized Risk-Free Return": risk_free_rate,
        "Sharpe Ratio": (annualized_return - risk_free_rate) / annualized_volatility
                        if volatility > 0 else np.nan,
        "Maximum Drawdown": float(-(daily["Equity"] / peak - 1).min()),
        "Number of Trades": len(trades),
        "Win Rate": trades["Net P/L ($)"].gt(0).mean() if len(trades) else np.nan,
        "Average P/L per Trade ($)": trades["Net P/L ($)"].mean(),
        "Average Return per Trade": trades["Net Return"].mean(),
        "Turnover (x)": daily["Turnover"].sum(),
        "Transaction Costs ($)": daily["Costs ($)"].sum(),
        "Costs / Initial Capital": daily["Costs ($)"].sum() / initial_capital,
    }


## 8. Verify the engine with synthetic data

These checks derive their signals, costs, capital, and holdout fixture dates from the configuration cell and never use market observations. After changing configuration, use **Restart Kernel and Run All Cells** to refresh all calculations and outputs. They cover delayed entry, stop blacklisting and reset, threshold inclusivity, end-of-window liquidation, two-leg costs, frozen holdings, no-pair cash windows, aggregation, and date-gated ingestion. A synthetic holdout row has an intentionally unusable price to verify it is never converted into model input.

In [ ]:
def run_synthetic_checks():
    import tempfile
    from unittest.mock import patch

    entry_z = (ENTRY_Z + STOP_Z) / 2
    stop_z = STOP_Z + (STOP_Z - ENTRY_Z) / 2
    rate = TRANSACTION_COST_BPS / 10_000
    capital = INITIAL_CAPITAL
    # Keep artificial price moves small even when testing large z thresholds.
    spread_std = 0.01 / max(1.0, stop_z)
    dates = pd.bdate_range("2020-01-02", periods=8)
    pair = {"A": "A", "B": "B", "Beta": 1.0,
            "Spread Mean": 0.0, "Spread Std": spread_std}
    def prices(zvalues):
        return pd.DataFrame({"A": 100 * np.exp(np.array(zvalues) * spread_std),
                             "B": 100.0}, index=dates[:len(zvalues)])
    formation_last = pd.Series({"A": 100.0, "B": 100.0}, name=dates[0] - pd.Timedelta(days=1))
    sample = prices([-entry_z, -entry_z, -stop_z, -entry_z, -entry_z, 0, -entry_z, -entry_z])
    daily, trades = backtest_pair(pair, sample, formation_last, capital)
    assert daily.iloc[0]["Position"] == 0  # First close creates, but cannot fill, the entry.
    assert trades.iloc[0]["Entry Date"] == dates[1]
    assert trades.iloc[0]["Exit Date"] == dates[3]
    assert trades.iloc[0]["Exit Reason"] == "Stop loss"
    assert len(trades) == 1 and daily.loc[dates[2]:, "Blacklisted"].all()
    assert daily.iloc[-1]["Position"] == 0
    np.testing.assert_allclose(daily.loc[dates[1]:dates[2], "Units A"], daily.loc[dates[1], "Units A"])
    np.testing.assert_allclose(daily["Costs ($)"], daily["Traded Value ($)"] * rate)
    np.testing.assert_allclose(daily["Equity"].iloc[-1] - capital, trades["Net P/L ($)"].sum())
    assert decide_order(1, -STOP_Z, False) == ((0, "Stop loss"), True)
    assert decide_order(-1, STOP_Z, False) == ((0, "Stop loss"), True)
    assert decide_order(1, -EXIT_Z, False)[0] == (0, "Mean reversion")
    assert decide_order(-1, EXIT_Z, False)[0] == (0, "Mean reversion")
    assert decide_order(0, -ENTRY_Z, False)[0] == (1, "Entry")
    assert decide_order(0, ENTRY_Z, False)[0] == (-1, "Entry")
    assert decide_order(0, -entry_z, True)[0] is None

    # A new invocation resets the blacklist; a formation signal may enter at the first close.
    seed = formation_last.copy()
    seed["A"] = 100 * np.exp(-entry_z * spread_std)
    flat_prices = prices([-entry_z] * 8)
    reset_daily, reset_trades = backtest_pair(pair, flat_prices, seed, capital)
    assert reset_trades.iloc[0]["Entry Date"] == dates[0]
    assert reset_trades.iloc[0]["Exit Reason"] == "End of window"
    assert not reset_daily["Blacklisted"].any()
    np.testing.assert_allclose(reset_trades["Costs ($)"].sum(), 2 * capital / (1 + rate) * rate)
    np.testing.assert_allclose(reset_daily["Equity"].iloc[-1], capital - reset_trades["Costs ($)"].sum())
    cash = combine_accounts([], dates, capital)
    assert cash["Net Return"].eq(0).all()
    combined = combine_accounts([daily, reset_daily], dates, (2 * capital))
    np.testing.assert_allclose(combined["Equity"], daily["Equity"] + reset_daily["Equity"])
    np.testing.assert_allclose((2 * capital) * (1 + combined["Net Return"]).cumprod(), combined["Equity"])
    with tempfile.TemporaryDirectory() as tmp:
        path = Path(tmp) / "fixture.csv"
        path.write_text(
            "Date,Adj Close\n"
            f"{(HOLDOUT_START - pd.Timedelta(days=1)):%Y-%m-%d},100\n"
            f"{HOLDOUT_START:%Y-%m-%d},DO_NOT_PARSE\n"
        )
        numeric = pd.to_numeric
        seen = []
        def record_conversion(value, **kwargs):
            seen.append(value)
            return numeric(value, **kwargs)
        with patch.object(pd, "to_numeric", side_effect=record_conversion):
            loaded = read_training_series(path)
        assert seen == ["100"] and loaded.index.max() < HOLDOUT_START
    print("Synthetic checks passed.")


run_synthetic_checks()

## 9. Run the training-only rolling experiment

Formation and selection use **membership-filtered** prices. Only after a window's pairs and parameters are fixed, load **raw adjusted closes for its selected stocks** to value and trade them throughout that window, including after index membership ends. Raw prices do not determine eligibility, correlations, regression fits, or direction selection. Their availability never changes selected pairs.

The raw reader applies the same training-only date gate. It caches training observations for selected stocks and then slices to the scheduled trading sessions. No holdout prices are converted or stored. Each window exports `trading_price_sources.csv`, including how many raw observations extend beyond filtered coverage and whether overlapping prices disagree. Missing raw trading observations still halt the run without dropping pairs or filling prices.

Each execution uses a new output directory. A manifest records `running`, `failed`, or `complete`; selection diagnostics are saved before trading, and a failed run does not emit a final performance summary. This approved raw-price policy replaces the earlier strict filtered-price run, which stopped on MUR/DVN in July–August 2017.

In [ ]:
def load_selected_raw_trading(selected, dates, cache):
    """Load training-only raw prices after selection; keep the requested session grid."""
    if dates.empty or dates.min() < DATA_START or dates.max() >= HOLDOUT_START:
        raise ValueError("Raw trading dates must remain inside the training partition")
    tickers = sorted(set(selected["A"]) | set(selected["B"]))
    for ticker in tickers:
        if ticker not in cache:
            path = RAW_DATA_FOLDER / f"{ticker}.csv"
            if not path.is_file():
                raise FileNotFoundError(f"Missing raw history for selected stock: {path}")
            cache[ticker] = read_training_series(path)
            assert cache[ticker].empty or cache[ticker].index.max() < HOLDOUT_START
    return pd.DataFrame({ticker: cache[ticker].reindex(dates) for ticker in tickers}, index=dates)


In [ ]:
def write_manifest(folder, status, **extra):
    record = {
        "status": status,
        "data_start": str(DATA_START.date()),
        "holdout_start": str(HOLDOUT_START.date()),
        "dataset_end_exclusive": str(DATA_END_EXCLUSIVE.date()),
        "formation_months": FORMATION_MONTHS, "trading_months": TRADING_MONTHS,
        "correlation_threshold": CORRELATION_THRESHOLD, "eg_significance": EG_SIGNIFICANCE,
        "entry_z": ENTRY_Z, "exit_z": EXIT_Z, "stop_z": STOP_Z,
        "cost_bps_per_leg": TRANSACTION_COST_BPS, "initial_capital": INITIAL_CAPITAL,
        "execution": "next session close; final scheduled close liquidation",
        "eligibility": "complete positive formation-session history",
        "direction_rule": "either EG direction passes; lowest EG p-value wins",
        "holdout_prices_used": False,
        "formation_price_source": "data/Filter_historical",
        "trading_price_source": "data/historical, selected pairs only",
        "versions": {package: version(package) for package in
                     ["numpy", "pandas", "statsmodels", "pandas_market_calendars"]},
        **extra,
    }
    (folder / "manifest.json").write_text(json.dumps(record, indent=2))


def missing_trading_observations(selected, trading):
    rows = []
    for _, pair in selected.iterrows():
        for ticker in [pair["A"], pair["B"]]:
            values = trading[ticker]
            bad = ~(np.isfinite(values) & values.gt(0))
            for date in values.index[bad]:
                rows.append({"A": pair["A"], "B": pair["B"], "Ticker": ticker, "Date": date})
    return pd.DataFrame(rows, columns=["A", "B", "Ticker", "Date"])

In [ ]:
def run_rolling_experiment(prices, schedule, mapping, folder, trading_loader=None):
    """Reselect, allocate, trade, and carry ending equity into the next window."""
    if prices.empty or prices.index.max() >= HOLDOUT_START:
        raise ValueError("The experiment accepts training prices only")
    capital = INITIAL_CAPITAL
    window_rows, daily_parts, trade_parts, pair_rows = [], [], [], []
    for _, window in schedule.iterrows():
        window_id = int(window["Window"])
        formation = interval(prices, window["Formation Start"], window["Formation End Exclusive"])
        trading = interval(prices, window["Trading Start"], window["Trading End Exclusive"])
        assert formation.index.max() < trading.index.min() < HOLDOUT_START
        window_folder = folder / f"window_{window_id:02d}"
        window_folder.mkdir()
        comparisons, eligibility = screen_correlations(formation, mapping)
        directions, selected = select_pairs(formation, comparisons)
        eligibility.to_csv(window_folder / "eligibility.csv", index=False)
        comparisons.to_csv(window_folder / "correlations.csv", index=False)
        directions.to_csv(window_folder / "eg_directions.csv", index=False)
        selected.to_csv(window_folder / "selected_pairs.csv", index=False)
        if trading_loader is not None:
            # Selection is already frozen. Raw prices never enter a formation slice.
            filtered_trading = trading
            trading = trading_loader(selected, filtered_trading.index)
            audit = []
            for ticker in trading.columns:
                filtered = filtered_trading[ticker]
                raw = trading[ticker]
                filtered_valid = np.isfinite(filtered) & filtered.gt(0)
                raw_valid = np.isfinite(raw) & raw.gt(0)
                shared = filtered_valid & raw_valid
                audit.append({
                    "Ticker": ticker, "Source": "data/historical",
                    "Trading Sessions": len(trading),
                    "Valid Raw Sessions": int(raw_valid.sum()),
                    "Sessions Beyond Filtered Coverage": int((raw_valid & ~filtered_valid).sum()),
                    "Overlapping Price Differences": int((~np.isclose(
                        raw.loc[shared], filtered.loc[shared], rtol=1e-7, atol=1e-8)).sum()),
                })
            pd.DataFrame(audit, columns=[
                "Ticker", "Source", "Trading Sessions", "Valid Raw Sessions",
                "Sessions Beyond Filtered Coverage", "Overlapping Price Differences",
            ]).to_csv(window_folder / "trading_price_sources.csv", index=False)
        missing = missing_trading_observations(selected, trading)
        if not missing.empty:
            missing.to_csv(window_folder / "missing_trading_prices.csv", index=False)
            raise ValueError(
                f"Window {window_id}: {len(missing)} missing/invalid selected-pair price observations. "
                f"See {window_folder / 'missing_trading_prices.csv'}. "
                "Restore training-period source data or explicitly choose a missing-price policy; "
                "selected pairs have not been dropped."
            )
        accounts, window_trades = [], []
        allocation = capital / len(selected) if len(selected) else 0.0
        for _, pair in selected.iterrows():
            daily, trades = backtest_pair(pair, trading, formation.iloc[-1], allocation)
            accounts.append(daily)
            trades.insert(0, "Window", window_id)
            window_trades.append(trades)
            pair_rows.append({"Window": window_id, "A": pair["A"], "B": pair["B"],
                              "Initial Capital": allocation,
                              **performance_metrics(daily, trades, allocation)})
            daily.to_csv(window_folder / f"{pair['A']}_{pair['B']}_daily.csv")
        trades = (pd.concat(window_trades, ignore_index=True) if window_trades
                  else pd.DataFrame(columns=["Window", *TRADE_COLUMNS]))
        portfolio = combine_accounts(accounts, trading.index, capital)
        metrics = performance_metrics(portfolio, trades, capital)
        window_rows.append({
            **window.to_dict(), "Initial Capital": capital,
            "Final Capital": portfolio["Equity"].iloc[-1],
            "Trading Sessions": len(trading), "Eligible Stocks": int(eligibility["Eligible"].sum()),
            "Compared Pairs": len(comparisons),
            "Correlation Passed": int(comparisons["Screen Passed"].sum()),
            "EG Directions Passed": int(directions["EG Passed"].eq(True).sum()),
            "Selected Pairs": len(selected),
            "Stop Exits": int(trades["Exit Reason"].eq("Stop loss").sum()),
            "Blacklisted Pairs": int(portfolio["Blacklisted Pairs"].iloc[-1]),
            **metrics,
        })
        portfolio.insert(0, "Window", window_id)
        portfolio.to_csv(window_folder / "portfolio_daily.csv")
        trades.to_csv(window_folder / "trades.csv", index=False)
        daily_parts.append(portfolio)
        trade_parts.append(trades)
        capital = float(portfolio["Equity"].iloc[-1])
        print(f"Window {window_id:02d}: {len(selected):3d} pairs, "
              f"net return {metrics['Cumulative Return']:+.2%}")
    full_daily = pd.concat(daily_parts)
    full_trades = pd.concat(trade_parts, ignore_index=True)
    # Costs must accumulate across windows for the full-path gross diagnostic.
    full_daily["Gross Equity"] = full_daily["Equity"] + full_daily["Costs ($)"].cumsum()
    return pd.DataFrame(window_rows), full_daily, full_trades, pd.DataFrame(pair_rows)

In [ ]:
raw_price_cache = {}  # New cache per run; never supplied to pair selection.
RUN_FOLDER = OUTPUT_ROOT / pd.Timestamp.now(tz="UTC").strftime("run_%Y%m%dT%H%M%S_%fZ")
RUN_FOLDER.mkdir(parents=True, exist_ok=False)
windows.to_csv(RUN_FOLDER / "windows.csv", index=False)
write_manifest(RUN_FOLDER, "running")
try:
    window_summary, full_daily, full_trades, pair_summary = run_rolling_experiment(
        training_prices, windows, gics, RUN_FOLDER,
        trading_loader=lambda selected, dates: load_selected_raw_trading(
            selected, dates, raw_price_cache,
        ),
    )
except Exception as error:
    write_manifest(RUN_FOLDER, "failed", error=str(error))
    raise

## 10. Chain the equity curve and reconcile results

Carry actual ending portfolio equity into the next window and redistribute it equally among that window's newly selected pairs. The compounded daily-return curve must equal the account-based equity curve. Recompute full-period statistics from this continuous daily curve and all completed trades; never average window Sharpes, drawdowns, or annualized returns.

In [ ]:
expected_dates = training_prices.index[training_prices.index >= windows.iloc[0]["Trading Start"]]
assert full_daily.index.equals(expected_dates)
assert full_daily.index.is_unique and full_daily.index.max() < HOLDOUT_START
chained_equity = INITIAL_CAPITAL * (1 + full_daily["Net Return"]).cumprod()
np.testing.assert_allclose(chained_equity, full_daily["Equity"], rtol=1e-10)
np.testing.assert_allclose(
    INITIAL_CAPITAL * (1 + window_summary["Cumulative Return"]).prod(),
    full_daily["Equity"].iloc[-1], rtol=1e-10,
)
np.testing.assert_allclose(full_daily["Costs ($)"].sum(), full_trades["Costs ($)"].sum(), atol=1e-7)
np.testing.assert_allclose(
    full_daily["Equity"].iloc[-1] - INITIAL_CAPITAL,
    full_trades["Net P/L ($)"].sum(), atol=1e-7,
)
assert (full_trades["Entry Signal Date"] < full_trades["Entry Date"]).all()
signaled_exits = full_trades["Exit Signal Date"].notna()
assert (full_trades.loc[signaled_exits, "Exit Signal Date"]
        < full_trades.loc[signaled_exits, "Exit Date"]).all()
full_summary = pd.DataFrame([performance_metrics(full_daily, full_trades, INITIAL_CAPITAL)],
                            index=["Full training trading period"])
print("Full-curve reconciliation passed.")

### Window and full-period metrics

The table reports each window separately, including its pair count and stop activity. Pair-level metrics and detailed traces are exported for inspection without printing thousands of trades in the notebook.

In [ ]:
percent_columns = ["Cumulative Return", "Gross Cumulative Return", "Annualized Return",
                   "Annualized Volatility", "Annualized Risk-Free Return", "Maximum Drawdown", "Win Rate",
                   "Average Return per Trade", "Costs / Initial Capital"]
formats = {column: "{:.2%}" for column in percent_columns}
formats.update({"Sharpe Ratio": "{:.2f}", "Turnover (x)": "{:.2f}",
                "Average P/L per Trade ($)": "${:,.2f}", "Transaction Costs ($)": "${:,.2f}"})
display(full_summary.style.format(formats, na_rep="—"))
window_display = ["Window", "Trading Start", "Trading End Exclusive", "Shortened",
                  "Selected Pairs", "Stop Exits", "Blacklisted Pairs", *full_summary.columns]
display(window_summary[window_display].style.format(formats, na_rep="—"))

In [ ]:
plot_equity = pd.concat([
    pd.Series([INITIAL_CAPITAL], index=[training_prices.index[training_prices.index < full_daily.index[0]][-1]]),
    full_daily["Equity"],
])
drawdown = plot_equity / plot_equity.cummax() - 1
fig, axes = plt.subplots(3, 1, figsize=(13, 10), sharex=True,
                         gridspec_kw={"height_ratios": [2, 1, 1]})
axes[0].plot(plot_equity.index, plot_equity, label="Net portfolio equity")
axes[0].set(ylabel="Equity ($)", title="Rolling strategy — training trading windows only")
axes[0].legend()
axes[1].fill_between(drawdown.index, drawdown * 100, 0, alpha=0.4, color="firebrick")
axes[1].set(ylabel="Drawdown (%)")
axes[2].step(window_summary["Trading Start"], window_summary["Selected Pairs"], where="post",
             label="Selected pairs")
axes[2].plot(full_daily.index, full_daily["Active Pairs"], alpha=0.6, label="Active pairs")
axes[2].set(ylabel="Pairs", xlabel="Trading date")
axes[2].legend()
for ax in axes:
    ax.grid(alpha=0.2)
fig.tight_layout()
fig.savefig(RUN_FOLDER / "equity_drawdown_pairs.png", dpi=160)
plt.show()

## 11. Export reproducible results

Outputs are confined to `outputs/rolling_training/run_<UTC timestamp>/`. The manifest reaches `complete` only after the full schedule, accounting checks, and final exports succeed. Older static validation outputs are not inputs to this experiment.

In [ ]:
window_summary.to_csv(RUN_FOLDER / "window_summary.csv", index=False)
full_summary.to_csv(RUN_FOLDER / "full_summary.csv")
full_daily.to_csv(RUN_FOLDER / "portfolio_daily.csv")
full_trades.to_csv(RUN_FOLDER / "trades.csv", index=False)
pair_summary.to_csv(RUN_FOLDER / "pair_summary.csv", index=False)
write_manifest(RUN_FOLDER, "complete", completed_windows=len(window_summary),
               first_trading_date=str(full_daily.index.min().date()),
               last_trading_date=str(full_daily.index.max().date()))
print(f"Saved completed training experiment to {RUN_FOLDER}")

## 12. Interpretation, limitations, and future validation

- **Equal-weight, independent positions:** overlapping pairs can repeatedly expose the portfolio to the same stocks, industries, and common risks. There are no portfolio-level exposure limits, covariance adjustments, or risk-parity sizing. Equal inception capital does not mean equal risk or continuously equal weights.
- **Selection uncertainty:** 5% is a per-direction EG threshold. Testing many pairs and choosing between two directions increases false discoveries; no multiple-testing correction is applied. EG assumes integrated log-price inputs; the requested workflow does not add separate integration-order screens.
- **Data limitations:** GICS classifications are static, available histories may omit failed/delisted stocks, and membership-filtered formation histories can end when membership ends. Selected pairs use raw trading prices for the entire scheduled window, so positions may continue after index removal. Complete formation-history eligibility can further favor established stocks. Missing selected raw trading observations cause a visible halt and require source-data resolution or an explicitly agreed execution policy.
- **Execution approximation:** adjusted closes represent return-adjusted units rather than an actual order book. Fixed beta sizes entry exposures; holdings remain fixed, so dollar weights drift. Next-close fills can cross the stop threshold. Borrow availability, borrow fees, financing, cash interest, market impact, and taxes are omitted; 0 bps per leg is the specified transaction-cost model.
- **Interpretation:** the first 12 months are formation-only. Short-window annualized statistics can be noisy, and all results here remain training-period research. Historical membership and price adjustments do not eliminate all data biases.

**Future validation:** after development is frozen, the same 12-month formation / two-month trading process can be applied to the reserved final three years. The first holdout formation period may use preceding training history; later windows may use only observations already historical at their decision time. This notebook does not load holdout prices, select pairs using them, execute that evaluation, or claim holdout performance.